In [ ]:
!pip -q install amplpy sympy numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 23.0 MB/s eta 0:00:00


In [ ]:
# 1. Instalación e inicialización de AMPL con tu licencia Community Edition
from amplpy import ampl_notebook

# Inicializa AMPL en Colab con tu UUID (Community Edition) Inicializar AMPL
ampl = ampl_notebook(
    modules=["highs", "cbc", "gurobi", "cplex"],  # solvers disponibles
    license_uuid="936b618d-a013-406f-9809-49679f557c26"
)

Licensed to AMPL Academic Community Edition License for <m.godoyseplveda@uandresbello.edu>.


In [ ]:
%%writefile hitmiss.mod
set X;                  # tamaños de lote 0..M
param C{X,X};           # costo esperado f(x1,x2)

var w{X,X} binary;      # 1 si se elige (x1,x2)

minimize TotalCost:
    sum{(i,j) in X cross X} C[i,j] * w[i,j];

s.t. ChooseOne:
    sum{(i,j) in X cross X} w[i,j] = 1;

# Para mostrar la politica
var x1;  var x2;
s.t. Defx1: x1 = sum{(i,j) in X cross X} i * w[i,j];
s.t. Defx2: x2 = sum{(i,j) in X cross X} j * w[i,j];


Overwriting hitmiss.mod


In [ ]:
%%bash
python - <<'PY'
import math, json

p_def = 1/3
K      = lambda x: 300 if x>0 else 0
penalty= 1600
M      = 6                      # 0..6 unidades por corrida

# --- construir matriz C[i,j] ---
def f2(x): return K(x)+100*x + (p_def**x)*penalty
tabla = { (i,j): K(i)+100*i + (p_def**i)*f2(j)
          for i in range(M+1) for j in range(M+1) }

# --- escribir .dat ---
with open('hitmiss.dat','w') as f:
    f.write('set X := ' + ' '.join(map(str,range(M+1))) + ' ;\n\n')
    f.write('param C : ' + ' '.join(map(str,range(M+1))) + ' :=\n')
    for i in range(M+1):
        fila = ' '.join(f'{tabla[(i,j)]:.5f}' for j in range(M+1))
        f.write(f'  {i}   {fila}\n')
    f.write(';\n')
print('Archivo hitmiss.dat generado.')
PY


Archivo hitmiss.dat generado.


In [ ]:
ampl.read('hitmiss.mod')
ampl.readData('hitmiss.dat')

ampl.option['solver'] = 'highs'   # o 'cbc'
ampl.solve()
ampl.display('x1','x2','TotalCost')


HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 573.25103
0 simplex iterations
0 branching nodes
x1 = 2
x2 = 3
TotalCost = 573.251

